In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!git clone https://github.com/Mugundh97B/codeswitch_speech_pipeline.git
%cd codeswitch_speech_pipeline

Cloning into 'codeswitch_speech_pipeline'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 47 (delta 2), reused 47 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 1.44 MiB | 4.55 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/codeswitch_speech_pipeline


In [3]:
!ls

pipeline.py  README.md	report	requirements.txt  src


In [6]:
from google.colab import files
files.upload()

Saving lid_model.pth to lid_model.pth


{'lid_model.pth': b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x12\x00\x10\x00lid_model/data.pklFB\x0c\x00ZZZZZZZZZZZZ\x80\x02ccollections\nOrderedDict\nq\x00)Rq\x01(X\x0e\x00\x00\x00model.0.weightq\x02ctorch._utils\n_rebuild_tensor_v2\nq\x03((X\x07\x00\x00\x00storageq\x04ctorch\nFloatStorage\nq\x05X\x01\x00\x00\x000q\x06X\x03\x00\x00\x00cpuq\x07M\xa0\x01tq\x08QK\x00K K\r\x86q\tK\rK\x01\x86q\n\x89h\x00)Rq\x0btq\x0cRq\rX\x0c\x00\x00\x00model.0.biasq\x0eh\x03((h\x04h\x05X\x01\x00\x00\x001q\x0fh\x07K tq\x10QK\x00K \x85q\x11K\x01\x85q\x12\x89h\x00)Rq\x13tq\x14Rq\x15X\x0e\x00\x00\x00model.2.weightq\x16h\x03((h\x04h\x05X\x01\x00\x00\x002q\x17h\x07M\x00\x02tq\x18QK\x00K\x10K \x86q\x19K K\x01\x86q\x1a\x89h\x00)Rq\x1btq\x1cRq\x1dX\x0c\x00\x00\x00model.2.biasq\x1eh\x03((h\x04h\x05X\x01\x00\x00\x003q\x1fh\x07K\x10tq QK\x00K\x10\x85q!K\x01\x85q"\x89h\x00)Rq#tq$Rq%X\x0e\x00\x00\x00model.4.weightq&h\x03((h\x04h\x05X\x01\x00\x00\x004q\'h\x07K tq

In [7]:
!pip install librosa soundfile torch torchaudio
!pip install git+https://github.com/openai/whisper.git
!pip install speechbrain

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-_7zo6k9e
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-_7zo6k9e
  Resolved https://github.com/openai/whisper.git to commit cba3768142a28276a90f14907e4900372c0c3ee0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=eaaf2116b9003b2c19afbefea65bdd3261d14f953b418e4e33298e878596d4d8
  Stored in directory: /tmp/pip-ephem-wheel-cache-l00bho42/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper


In [8]:
import whisper

# Load model
model = whisper.load_model("base")

# Transcribe
result = model.transcribe("original_segment.wav")

# Print first few lines
for seg in result["segments"][:5]:
    print(seg["start"], "-", seg["end"], ":", seg["text"])


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 95.5MiB/s]


0.0 - 2.9 :  چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa
2.9 - 5.86 :  ڈھフリックス ھا ہوں بہت سب کے پاس
5.86 - 8.14 :  ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے
8.14 - 9.86 :  ایک فیچھے ہوں ہوں ہوں
9.86 - 11.3 :  جب ہی آپ ایک


In [9]:
# Save full transcript
with open("transcript.txt", "w") as f:
    f.write(result["text"])

print("Transcript saved!")

Transcript saved!


In [10]:
!ls

lid_model.pth	      README.md		src
original_segment.wav  report		student_voice_ref.wav
pipeline.py	      requirements.txt	transcript.txt


In [11]:
import librosa
import numpy as np
import torch
import torch.nn as nn

# === Feature Extraction ===
def extract_mfcc_frames(audio_path, sr=16000, frame_size=0.5):
    y, sr = librosa.load(audio_path, sr=sr)

    frame_length = int(sr * frame_size)
    frames = []

    for i in range(0, len(y), frame_length):
        frame = y[i:i+frame_length]

        if len(frame) < frame_length:
            continue

        mfcc = librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)

        frames.append(mfcc_mean)

    return np.array(frames)


# === Model ===
class LIDModel(nn.Module):
    def __init__(self):
        super(LIDModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(13, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.model(x)


# === Load features ===
X = extract_mfcc_frames("original_segment.wav")
X = torch.tensor(X, dtype=torch.float32)

# === Load trained model  ===
model = LIDModel()
model.load_state_dict(torch.load("lid_model.pth", map_location=torch.device('cpu')))
model.eval()

# === Predict ===
with torch.no_grad():
    outputs = model(X)
    preds = torch.argmax(outputs, dim=1)

print("First 20 predictions:")
print(preds[:20])

First 20 predictions:
tensor([0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0])


In [12]:
# Convert predictions to timeline

frame_duration = 0.5  # same as used earlier
timeline = []

for i, label in enumerate(preds):
    start = i * frame_duration
    end = start + frame_duration

    lang = "English" if label.item() == 1 else "Hindi"

    timeline.append((start, end, lang))

# Print first 10
print("First 10 timeline entries:\n")
for t in timeline[:10]:
    print(t)

First 10 timeline entries:

(0.0, 0.5, 'Hindi')
(0.5, 1.0, 'English')
(1.0, 1.5, 'Hindi')
(1.5, 2.0, 'Hindi')
(2.0, 2.5, 'Hindi')
(2.5, 3.0, 'Hindi')
(3.0, 3.5, 'Hindi')
(3.5, 4.0, 'English')
(4.0, 4.5, 'Hindi')
(4.5, 5.0, 'Hindi')


In [13]:
# Load your vocab (recreate here)
vocab = [
    "learning", "machine", "signal", "system",
    "probability", "distribution", "feature",
    "model", "training", "data", "models", "flims", "train", "data", "movies"
]

def boost_text(text, vocab):
    words = text.split()
    new_words = []

    for w in words:
        lw = w.lower()

        replaced = False
        for v in vocab:
            if lw[:4] == v[:4]:  # prefix match
                new_words.append(v)
                replaced = True
                break

        if not replaced:
            new_words.append(w)

    return " ".join(new_words)


# Apply to full transcript
with open("transcript.txt") as f:
    transcript = f.read()

boosted = boost_text(transcript, vocab)

print("Original:\n", transcript[:200])
print("\nBoosted:\n", boosted[:200])

Original:
  چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa ڈھフリックス ھا ہوں بہت سب کے پاس ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے ایک فیچھے ہوں ہوں ہوں جب ہی آپ ایک ایک چین مووی دیکھتے ہیں تو ایک چین مووی اگر آپ 

Boosted:
 چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa ڈھフリックス ھا ہوں بہت سب کے پاس ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے ایک فیچھے ہوں ہوں ہوں جب ہی آپ ایک ایک چین مووی دیکھتے ہیں تو ایک چین مووی اگر آپ د


In [14]:
!pip install eng_to_ipa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 48.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for eng_to_ipa: filename=eng_to_ipa-0.0.2-py3-none-any.whl size=2822601 sha256=adbbb9ebae2dc83c9f2106bf7c2ed8e5e8b792ed39b19f6f5937ca76e6715e14
  Stored in directory: /root/.cache/pip/wheels/cf/2b/91/f066669a92e46921f2c953cb87c8f714e2c127cb9342525d54
Successfully built eng_to_ipa


In [15]:
import eng_to_ipa as ipa

def is_english(word):
    return all(c.isascii() for c in word)


def convert_to_ipa(text):
    words = text.split()
    ipa_words = []

    for w in words:
        if is_english(w):
            ipa_word = ipa.convert(w)
        else:
            ipa_word = w  # fallback for Hindi

        ipa_words.append(ipa_word)

    return " ".join(ipa_words)


# Load transcript
with open("transcript.txt") as f:
    transcript = f.read()

ipa_text = convert_to_ipa(transcript)

print("Original:\n", transcript[:200])
print("\nIPA:\n", ipa_text[:200])

Original:
  چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa ڈھフリックス ھا ہوں بہت سب کے پاس ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے ایک فیچھے ہوں ہوں ہوں جب ہی آپ ایک ایک چین مووی دیکھتے ہیں تو ایک چین مووی اگر آپ 

IPA:
 چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa ڈھフリックス ھا ہوں بہت سب کے پاس ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے ایک فیچھے ہوں ہوں ہوں جب ہی آپ ایک ایک چین مووی دیکھتے ہیں تو ایک چین مووی اگر آپ د


In [17]:
# Simple Tamil dictionary (expandable)
tamil_dict = {
    "i": "நான்",
    "hope": "நம்புகிறேன்",
    "you": "நீங்கள்",
    "all": "அனைவரும்",
    "have": "உள்ளது",
    "seen": "பார்த்தீர்கள்",
    "many": "பல",
    "movies": "திரைப்படங்கள்"
}

def translate(text):
    words = text.lower().split()
    out = []

    for w in words:
        clean = w.strip(".,!?")

        if clean in tamil_dict:
            out.append(tamil_dict[clean])
        else:
            out.append(w)

    return " ".join(out)


# Load transcript
with open("transcript.txt") as f:
    transcript = f.read()

tamil_text = translate(transcript)

print("Original:\n", transcript[:200])
print("\nTamil:\n", tamil_text[:200])

Original:
  چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa ڈھフリックス ھا ہوں بہت سب کے پاس ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے ایک فیچھے ہوں ہوں ہوں جب ہی آپ ایک ایک چین مووی دیکھتے ہیں تو ایک چین مووی اگر آپ 

Tamil:
 چھے میں کچھ ڈکھا تھا ہوں پہلی ڈکھا تھا ہوں گa ڈھフリックス ھا ہوں بہت سب کے پاس ڈھフリックス ہوں آپ بہت سارے موویز دیکھتایا ہے ایک فیچھے ہوں ہوں ہوں جب ہی آپ ایک ایک چین مووی دیکھتے ہیں تو ایک چین مووی اگر آپ د


In [18]:
import librosa
import numpy as np

# Load ORIGINAL lecture (this is why we uploaded it)
y, sr = librosa.load("original_segment.wav", sr=16000)

# === F0 (pitch) extraction ===
f0, voiced_flag, voiced_probs = librosa.pyin(
    y,
    fmin=librosa.note_to_hz('C2'),
    fmax=librosa.note_to_hz('C7')
)

# === Energy extraction ===
energy = np.sqrt(np.mean(y**2))

print("F0 length:", len(f0))
print("Sample F0 values:", f0[:10])
print("Energy:", energy)

F0 length: 18751
Sample F0 values: [135.42588547 135.42588547 135.42588547 134.64588976 135.42588547
 156.464662   136.21039964 136.21039964 136.21039964 136.99945846]
Energy: 0.06380418


In [19]:
import librosa

# Clean NaNs from F0
f0_clean = f0[~np.isnan(f0)]

# Create a dummy target (slightly modified version)
f0_target = f0_clean * 1.1  # simulate new speech

# Apply DTW
D, wp = librosa.sequence.dtw(f0_clean.reshape(1, -1),
                            f0_target.reshape(1, -1))

print("DTW cost matrix shape:", D.shape)
print("Warping path length:", len(wp))

DTW cost matrix shape: (13513, 13513)
Warping path length: 18728


In [20]:
!pip install gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [21]:
from gtts import gTTS

# Use your translated Tamil text
text = "நான் நம்புகிறேன் நீங்கள் புரிந்து கொண்டு இருக்கிறீர்கள்"

tts = gTTS(text=text, lang='ta')
tts.save("synthetic_tamil.wav")

print("Synthetic audio created!")

Synthetic audio created!


In [22]:
import librosa
import numpy as np

def extract_features(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    # Spectral centroid
    spec_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

    # Zero-crossing rate
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    features = np.concatenate([mfcc_mean, [spec_centroid, zcr]])

    return features


# REAL
real_feat = extract_features("student_voice_ref.wav")

# FAKE
fake_feat = extract_features("synthetic_tamil.wav")

print("Real features shape:", real_feat.shape)
print("Fake features shape:", fake_feat.shape)

Real features shape: (15,)
Fake features shape: (15,)


In [23]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# Prepare dataset
X = np.vstack([real_feat, fake_feat])

# Labels: 0 = real, 1 = fake
y = np.array([0, 1])

# Train classifier
clf = LogisticRegression()
clf.fit(X, y)

# Predict
pred = clf.predict(X)

print("Predictions:", pred)
print("Expected   :", y)

Predictions: [0 1]
Expected   : [0 1]


In [24]:
!ls


lid_model.pth	      README.md		src		       transcript.txt
original_segment.wav  report		student_voice_ref.wav
pipeline.py	      requirements.txt	synthetic_tamil.wav


In [29]:
!pip freeze > requirements.txt